In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

def sql(requete):
    """Exécute une requête Spark SQL et affiche le résultat en tableau."""
    return spark.sql(requete).toPandas()

In [2]:
sql("SHOW TABLES IN lake.bronze")

,namespace,tableName,isTemporary
0,bronze,accounts,False
1,bronze,customers,False
2,bronze,fraud_reports,False
3,bronze,transactions,False


In [3]:
sql("""
SELECT category, COUNT(*) AS nb, ROUND(SUM(try_cast(amount AS DOUBLE)), 2) AS total
FROM lake.bronze.transactions
GROUP BY category ORDER BY total DESC
""")

,category,nb,total
0,revenu,515,1530270.68
1,voyage,520,592032.31
2,shopping,1675,520521.92
3,alimentation,3952,240201.72
4,transport,2738,111486.04
5,loisirs,1327,102465.05
6,restaurant,2002,98821.27
7,sante,728,75613.23
8,REVENU,1,3070.11
9,SHOPPING,12,2154.71


In [4]:
sql("""
SELECT snapshot_id, committed_at, summary['added-records'] AS lignes_ajoutees
FROM lake.bronze.transactions.snapshots
ORDER BY committed_at;
""")

,snapshot_id,committed_at,lignes_ajoutees
0,2681416204403164066,2026-10-11 14:53:31.100,1


In [5]:
sql("""
SELECT COUNT(*) AS nb_fichiers, SUM(record_count) AS nb_lignes
FROM lake.bronze.transactions.files;
""")

,nb_fichiers,nb_lignes
0,1380,13544


In [6]:
# le tout premier snapshot de la table
premier = spark.sql("""
    SELECT snapshot_id FROM lake.bronze.transactions.snapshots
    ORDER BY committed_at LIMIT 1
""").first()[0]

print("Snapshot :", premier)
sql(f"SELECT COUNT(*) AS nb_alors FROM lake.bronze.transactions VERSION AS OF {premier}")

Snapshot : 2681416204403164066


,nb_alors
0,13544


In [8]:
sql("""
SELECT SUM(amount) FROM lake.bronze.transactions;
""")


{"ts": "2026-10-18 17:32:08.769", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '39,3' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 2, position 8) ==\nSELECT SUM(amount) FROM lake.bronze.transactions;\n       ^^^^^^^^^^^\n\n\nJVM stacktrace:\norg.apache.spark.SparkNumberFormatException\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:147)\n\tat org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala:-1)\n\tat org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregate_sum_0$(:-1)\n\tat org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_d

NumberFormatException: [CAST_INVALID_INPUT] The value '39,3' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 2, position 8) ==
SELECT SUM(amount) FROM lake.bronze.transactions;
       ^^^^^^^^^^^


JVM stacktrace:
org.apache.spark.SparkNumberFormatException
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:147)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala:-1)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregate_sum_0$(:-1)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doConsume_0$(:-1)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIter

In [ ]:
sql("""
SELECT ROUND(SUM(try_cast(amount AS DOUBLE)), 2) FROM lake.bronze.transactions;
""")



,"round(sum(TRY_CAST(amount AS DOUBLE)), 2)"
0,2832043.96


In [ ]:
sql("""
SELECT transaction_id, amount FROM lake.bronze.transactions
WHERE try_cast(amount AS DOUBLE) IS NULL OR try_cast(amount AS DOUBLE)<=0
""")

,transaction_id,amount
0,T00011062,-26.18
1,T00011073,-40.64
2,T00007330,"39,3"
3,T00010817,"41,77"
4,T00010817,"41,77"
...,...,...
176,T00011112,-109.06
177,T00010323,"71,69"
178,T00010320,"19,96"
179,T00007794,-575.94


In [ ]:
sql("""
SELECT category, COUNT(*) FROM lake.bronze.transactions GROUP BY category ORDER BY category;
""")

,category,count(1)
0,ALIMENTATION,25
1,LOISIRS,5
2,RESTAURANT,13
3,REVENU,1
4,SANTE,4
5,SHOPPING,11
6,TRANSPORT,17
7,VOYAGE,1
8,alimentation,3490
9,loisirs,1172


In [ ]:
sql("""
SELECT country, COUNT(*) FROM lake.bronze.transactions GROUP BY country ORDER BY country;
""")

,country,count(1)
0,BE,140
1,BR,35
2,CN,38
3,DE,140
4,ES,123
5,FR,11315
6,IT,144
7,NG,34
8,RU,30
9,TH,40


In [ ]:
sql("""
SELECT transaction_id, category FROM lake.bronze.transactions WHERE merchant IS NULL;
""")

,transaction_id,category
0,T00006880,shopping
1,T00008539,alimentation
2,T00008535,restaurant
3,T00006924,transport
4,T00007355,restaurant
...,...,...
72,T00005270,shopping
73,T00011830,alimentation
74,T00005926,alimentation
75,T00005686,alimentation


In [ ]:
sql("""
SELECT transaction_id, timestamp FROM lake.bronze.transactions
WHERE try_to_timestamp(timestamp) IS NULL;
""")

,transaction_id,timestamp
0,T00006605,11/10/2026 13:55
1,T00009426,11/10/2026 14:18
2,T00012272,11/10/2026 14:41
3,T00005245,11/10/2026 13:43
4,T00009595,11/10/2026 14:19
...,...,...
75,T00004787,11/10/2026 13:39
76,T00008471,11/10/2026 14:10
77,T00011111,11/10/2026 14:32
78,T00006951,11/10/2026 13:57


In [ ]:
sql("""
SELECT transaction_id, COUNT(*) AS nb FROM lake.bronze.transactions
GROUP BY transaction_id HAVING COUNT(*) > 1;
""")

,transaction_id,nb
0,T00011767,2
1,T00005765,2
2,T00005389,2
3,T00009435,2
4,T00010591,2
...,...,...
107,T00011193,2
108,T00011583,2
109,T00011577,2
110,T00007322,2


In [ ]:
sql("""
SELECT t.transaction_id, t.account_id FROM lake.bronze.transactions t
LEFT JOIN lake.bronze.accounts a ON t.account_id = a.account_id
WHERE a.account_id IS NULL;
""")

,transaction_id,account_id
0,T00005406,A99999
1,T00009262,A99999
2,T00008565,A99999
3,T00010033,A99999
4,T00008596,A99999
...,...,...
89,T00010836,A99999
90,T00009759,A99999
91,T00005589,A99999
92,T00005985,A99999


In [ ]:
sql("""
SELECT raw_value, kafka_offset FROM lake.bronze.transactions WHERE transaction_id IS NULL;
""")

,raw_value,kafka_offset
0,"{""transaction_id"": ""TX-BUG""",4839


In [ ]:
sql("""
SELECT kafka_partition, kafka_offset, COUNT(*) FROM lake.bronze.transactions
GROUP BY kafka_partition, kafka_offset HAVING COUNT(*) > 1;
""")

,kafka_partition,kafka_offset,count(1)


In [ ]:
sql("""
SELECT transaction_id, kafka_partition, kafka_offset, amount, ingested_at
FROM lake.bronze.transactions
WHERE transaction_id IN (
    SELECT transaction_id FROM lake.bronze.transactions
    GROUP BY transaction_id HAVING COUNT(*) > 1
)
ORDER BY transaction_id, kafka_offset
""")

,transaction_id,kafka_partition,kafka_offset,amount,ingested_at
0,T00000120,1,34,80.52,2026-10-11 13:35:39.953
1,T00000120,1,35,80.52,2026-10-11 13:35:39.953
2,T00000127,0,46,5.54,2026-10-11 13:35:39.953
3,T00000127,0,47,5.54,2026-10-11 13:35:39.953
4,T00000285,0,115,89.52,2026-10-11 13:35:39.953
...,...,...,...,...,...
233,T00013348,2,4856,152.43,2026-10-11 14:50:50.006
234,T00013404,2,4873,66.4,2026-10-11 14:51:10.005
235,T00013404,2,4874,66.4,2026-10-11 14:51:10.005
236,T00013522,1,3770,65.55,2026-10-11 14:52:20.007


In [ ]:
sql("""
SELECT
  SUM(CASE WHEN amount LIKE '%,%' THEN 1 ELSE 0 END)                                AS montant_texte,
  SUM(CASE WHEN try_cast(amount AS DOUBLE) <= 0 THEN 1 ELSE 0 END)                 AS montant_impossible,
  SUM(CASE WHEN category <> lower(trim(category)) THEN 1 ELSE 0 END)               AS casse_categorie,
  SUM(CASE WHEN country <> upper(trim(country)) THEN 1 ELSE 0 END)                 AS casse_pays,
  SUM(CASE WHEN transaction_id IS NOT NULL AND merchant IS NULL THEN 1 ELSE 0 END) AS commercant_manquant,
  SUM(CASE WHEN transaction_id IS NOT NULL AND try_to_timestamp(timestamp) IS NULL
           THEN 1 ELSE 0 END)                                                      AS date_autre_format,
  COUNT(transaction_id) - COUNT(DISTINCT transaction_id)                           AS doublons,
  SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END)                          AS message_illisible
FROM lake.bronze.transactions
""")


,montant_texte,montant_impossible,casse_categorie,casse_pays,commercant_manquant,date_autre_format,doublons,message_illisible
0,96,102,86,91,82,84,119,1


In [ ]:
sql("""
SELECT COUNT(*) AS compte_inconnu
FROM lake.bronze.transactions t
LEFT JOIN lake.bronze.accounts a ON t.account_id = a.account_id
WHERE t.account_id IS NOT NULL AND a.account_id IS NULL
""")


,compte_inconnu
0,95


In [ ]:
sql("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN amount LIKE '%,%' THEN 1 ELSE 0 END) AS montant_texte,
  SUM(CASE WHEN category <> lower(trim(category)) THEN 1 ELSE 0 END) AS casse_categorie,
  SUM(CASE WHEN transaction_id IS NOT NULL
           AND try_to_timestamp(timestamp) IS NULL THEN 1 ELSE 0 END) AS date_autre,
  COUNT(transaction_id) - COUNT(DISTINCT transaction_id) AS doublons
FROM lake.bronze.transactions
""")


,total,montant_texte,casse_categorie,date_autre,doublons
0,13544,96,86,84,119


In [ ]:
sql("""
SELECT
  (SELECT COUNT(*) FROM lake.bronze.transactions)        AS bronze,
  (SELECT COUNT(*) FROM lake.silver.transactions)        AS silver,
  (SELECT COUNT(*) FROM lake.silver.transactions_rejetees) AS rejetees
""")


,bronze,silver,rejetees
0,13544,13228,316


In [ ]:
sql("SELECT motif_rejet, COUNT(*) AS nb FROM lake.silver.transactions_rejetees GROUP BY motif_rejet")

,motif_rejet,nb
0,message_illisible,1
1,doublon,119
2,montant_impossible,101
3,compte_inconnu,95


In [ ]:
sql("SELECT category, COUNT(*) AS nb FROM lake.silver.transactions GROUP BY category ORDER BY category")

,category,nb
0,alimentation,3892
1,loisirs,1302
2,restaurant,1968
3,revenu,500
4,sante,711
5,shopping,1651
6,transport,2697
7,voyage,507


In [ ]:
sql("SELECT * FROM lake.gold.depenses_par_categorie ORDER BY total DESC")



,category,nb_transactions,total,montant_moyen
0,voyage,492,580456.36,1179.79
1,shopping,1612,510862.35,316.91
2,alimentation,3814,237993.66,62.40
3,transport,2651,110448.12,41.66
4,loisirs,1277,100945.33,79.05
5,restaurant,1931,97085.23,50.28
6,sante,693,73776.98,106.46


In [ ]:
sql("SELECT est_fraude, COUNT(*) AS nb FROM lake.gold.transactions_ml GROUP BY est_fraude")

,est_fraude,nb
0,1,45
1,0,12683
